In [1]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot"

os.environ["HADOOP_HOME"] = r"C:\hadoop"

os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
print(
    "winutils exists =",
    os.path.exists(r"C:\hadoop\bin\winutils.exe"),
)
print(
    "hadoop.dll exists =",
    os.path.exists(r"C:\hadoop\bin\hadoop.dll"),
)

JAVA_HOME = C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot
HADOOP_HOME = C:\hadoop
winutils exists = False
hadoop.dll exists = False


In [8]:
from pathlib import Path
import os

if "project_path" not in globals():
    project_path = Path.cwd().parent
    os.chdir(project_path)

print("Project path:", project_path)

Project path: c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk


In [2]:
%load_ext autoreload
%autoreload 2
from __future__ import annotations

from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

from credit_risk.utils.spark import create_spark_session
from credit_risk.pipelines.data_preprocess import (

    create_path,
    read_spark_parquet,
    build_master_dataset_spark,
    build_performance_spark,
)

from credit_risk.features.origination_spark import (
    build_origination_spark,
)

from credit_risk.features.behavioral_spark import (
    add_calculated_loan_age_spark,
    add_prior_serious_delinquency_flag_spark,
    build_behavioral_features_spark,
)

from credit_risk.target.behavioral_spark import (
    build_behavioral_target_spark,
)
from credit_risk.utils.config import read_config

In [4]:
project_path = Path.cwd().parent

In [5]:
config = read_config(project_path)

In [11]:
os.getcwd()

'c:\\Users\\vorad\\OneDrive\\Desktop\\Projects\\mortgage-credit-risk'

In [12]:
config['parameters']['evaluation']

{'skip': False,
 'mode': 'existing_model',
 'model': {'version': 'test_unbalanced_quarterly', 'type': 'xgboost'},
 'threshold_selection': {'enabled': True,
  'optimisation_metric': 'f1',
  'candidate_thresholds_range': [0.01, 0.3],
  'candidate_thresholds_step': 0.01},
 'datasets': {'validation': True, 'oot': True},
 'shap': {'enabled': False},
 'classification': {'threshold': 0.5},
 'risk': {'n_deciles': 10},
 'calibration': {'method': 'beta',
  'bins': [[0.0, 0.01],
   [0.01, 0.02],
   [0.02, 0.05],
   [0.05, 0.1],
   [0.1, 0.2],
   [0.2, 0.5],
   [0.5, 1.0]]}}

In [13]:
def add_calculated_loan_age_spark(
    df: DataFrame,
) -> DataFrame:
    """
    Calculate loan age from the canonical monthly Period[M] ordinal.

    Pandas reads the Parquet fields as Period[M], e.g.:

        2015-05

    Spark reads the underlying monthly ordinal, e.g.:

        544

    The Pandas calculation:

        months(period - first_payment_date) + 1

    is therefore equivalent to:

        period - first_payment_date + 1
    """

    # _validate_required_columns(
    #     df,
    #     {
    #         "period",
    #         "first_payment_date",
    #     },
    #     "calculated loan age",
    # )

    return df.withColumn(
        "calculated_loan_age",
        (F.col("period") - F.col("first_payment_date") + F.lit(1)).cast("int"),
    )

In [14]:
spark = create_spark_session(config)

origination_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "origination_path",
    "freddie_mac",
    2015,
)

performance_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "performance_path",
    "freddie_mac",
    2015,
)
origination_df = spark.read.parquet(str(os.getcwd()/ origination_path))

performance_df = spark.read.parquet(str(os.getcwd() / performance_path))

origination_features = build_origination_spark(
    origination_df,
    config,
)

origination_dates = origination_df.select(
    "loan_id",
    "first_payment_date",
).dropDuplicates(
    ["loan_id"],
)

origination_features = origination_features.join(
    origination_dates,
    on="loan_id",
    how="left",
)

performance_features = build_performance_spark(
    performance_df,
)

master = build_master_dataset_spark(
    origination_features,
    performance_features,
)

master = add_calculated_loan_age_spark(
    master,
)

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
master.select(
    "period",
    "first_payment_date",
).show(20, truncate=False)

+------+------------------+
|period|first_payment_date|
+------+------------------+
|551   |552               |
|552   |552               |
|553   |552               |
|554   |552               |
|555   |552               |
|556   |552               |
|557   |552               |
|558   |552               |
|559   |552               |
|560   |552               |
|561   |552               |
|562   |552               |
|563   |552               |
|564   |552               |
|565   |552               |
|566   |552               |
|567   |552               |
|568   |552               |
|569   |552               |
|570   |552               |
+------+------------------+
only showing top 20 rows


In [ ]:
ages = config["parameters"]["behavioral"]["observation_ages"]

print("Configured observation ages:")
print(ages)

print("\nCalculated loan age distribution:")

(
    master.groupBy("calculated_loan_age")
    .count()
    .orderBy("calculated_loan_age")
    .show(40, truncate=False)
)

print("\nRows at configured observation ages:")

(
    master.filter(F.col("calculated_loan_age").isin(ages))
    .groupBy("calculated_loan_age")
    .count()
    .orderBy("calculated_loan_age")
    .show()
)

Configured observation ages:
[6, 12]

Calculated loan age distribution:
+-------------------+-------+
|calculated_loan_age|count  |
+-------------------+-------+
|-1                 |18     |
|0                  |1292003|
|1                  |1394504|
|2                  |1421853|
|3                  |1424938|
|4                  |1422576|
|5                  |1418727|
|6                  |1412959|
|7                  |1405045|
|8                  |1395671|
|9                  |1385834|
|10                 |1375332|
|11                 |1364054|
|12                 |1351990|
|13                 |1338978|
|14                 |1325468|
|15                 |1312232|
|16                 |1299312|
|17                 |1286675|
|18                 |1274142|
|19                 |1261977|
|20                 |1250380|
|21                 |1239683|
|22                 |1229460|
|23                 |1219607|
|24                 |1209547|
|25                 |1198838|
|26                 |1188079

In [8]:
behavioral_features = build_behavioral_features_spark(
    master,
    config,
)

In [9]:
print(
    "Behavioral feature columns:",
    len(behavioral_features.columns),
)

print(
    "Behavioral feature rows:",
    behavioral_features.count(),
)

Behavioral feature columns: 44
Behavioral feature rows: 2741661


In [10]:
target = build_behavioral_target_spark(
    master,
    config,
)

In [11]:
(target.groupBy("observation_age").count().orderBy("observation_age").show())

+---------------+-------+
|observation_age|  count|
+---------------+-------+
|              6|1404307|
|             12|1338616|
+---------------+-------+



In [12]:
(target.groupBy("future_90dpd_12m").count().orderBy("future_90dpd_12m").show())

+----------------+-------+
|future_90dpd_12m|  count|
+----------------+-------+
|               0|2733840|
|               1|   9083|
+----------------+-------+



In [13]:
(
    behavioral_features.groupBy("observation_age")
    .count()
    .orderBy("observation_age")
    .show()
)

+---------------+-------+
|observation_age|  count|
+---------------+-------+
|              6|1404425|
|             12|1337236|
+---------------+-------+



In [14]:
modelling = behavioral_features.join(
    target,
    on=[
        "loan_id",
        "observation_age",
    ],
    how="inner",
)

print("Final modelling rows:", modelling.count())

(modelling.groupBy("observation_age").count().orderBy("observation_age").show())

Final modelling rows: 2741047
+---------------+-------+
|observation_age|  count|
+---------------+-------+
|              6|1403922|
|             12|1337125|
+---------------+-------+



In [15]:
(modelling.groupBy("future_90dpd_12m").count().orderBy("future_90dpd_12m").show())

+----------------+-------+
|future_90dpd_12m|  count|
+----------------+-------+
|               0|2733454|
|               1|   7593|
+----------------+-------+



In [16]:
# 1. Grain
duplicate_keys = (
    modelling.groupBy("loan_id", "observation_age").count().filter(F.col("count") > 1)
)

print("Duplicate keys:", duplicate_keys.count())

Duplicate keys: 0


In [17]:
# 2. Lifecycle consistency
invalid_age = modelling.filter(F.col("calculated_loan_age") != F.col("observation_age"))

print("Invalid calculated ages:", invalid_age.count())

Invalid calculated ages: 0


In [18]:
# 3. Final event rate
(
    modelling.groupBy("observation_age")
    .agg(
        F.count("*").alias("rows"),
        F.sum("future_90dpd_12m").alias("events"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy("observation_age")
    .show()
)

+---------------+-------+------+--------------------+
|observation_age|   rows|events|          event_rate|
+---------------+-------+------+--------------------+
|              6|1403922|  3105| 0.00221166133161244|
|             12|1337125|  4488|0.003356455080863...|
+---------------+-------+------+--------------------+



In [29]:
print("Rows at configured ages:")

ages = [6,12]
rows_at_age = master.filter(F.col("calculated_loan_age").isin(ages)).count()

print(rows_at_age)

print("\nRows after termination filter:")

rows_not_terminated = (
    master.filter(F.col("calculated_loan_age").isin(ages))
    .filter(F.col("zero_balance_code").isNull())
    .count()
)

print(rows_not_terminated)

Rows at configured ages:
2764949

Rows after termination filter:
2743537


In [10]:
sample = (
    master.select(
        "loan_id",
        "period",
        "first_payment_date",
    )
    .limit(30)
    .collect()
)

for row in sample:
    print(row)

Row(loan_id='F15Q40000001', period=551, first_payment_date=552)
Row(loan_id='F15Q40000001', period=552, first_payment_date=552)
Row(loan_id='F15Q40000001', period=553, first_payment_date=552)
Row(loan_id='F15Q40000001', period=554, first_payment_date=552)
Row(loan_id='F15Q40000001', period=555, first_payment_date=552)
Row(loan_id='F15Q40000001', period=556, first_payment_date=552)
Row(loan_id='F15Q40000001', period=557, first_payment_date=552)
Row(loan_id='F15Q40000001', period=558, first_payment_date=552)
Row(loan_id='F15Q40000001', period=559, first_payment_date=552)
Row(loan_id='F15Q40000001', period=560, first_payment_date=552)
Row(loan_id='F15Q40000001', period=561, first_payment_date=552)
Row(loan_id='F15Q40000001', period=562, first_payment_date=552)
Row(loan_id='F15Q40000001', period=563, first_payment_date=552)
Row(loan_id='F15Q40000001', period=564, first_payment_date=552)
Row(loan_id='F15Q40000001', period=565, first_payment_date=552)
Row(loan_id='F15Q40000001', period=566, 

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

origination_path = Path("data/02_intermediate/freddie_mac/2015/origination.parquet")

origination_raw = spark.read.parquet(str(origination_path))

print("Columns:", len(origination_raw.columns))
print(origination_raw.columns)

Columns: 31
['credit_score', 'first_payment_date', 'first_time_homebuyer_flag', 'maturity_date', 'msa', 'mi_percentage', 'number_of_units', 'occupancy_status', 'original_cltv', 'original_dti', 'original_upb', 'original_ltv', 'original_interest_rate', 'channel', 'prepayment_penalty_flag', 'amortization_type', 'property_state', 'property_type', 'postal_code', 'loan_id', 'loan_purpose', 'original_loan_term', 'number_of_borrowers', 'seller_name', 'super_conforming_flag', 'pre_harp_loan_id', 'special_eligibility_program', 'harp_indicator', 'property_valuation_method', 'interest_only_indicator', 'vantage_score_4']


In [12]:
origination_raw.printSchema()

root
 |-- credit_score: long (nullable = true)
 |-- first_payment_date: long (nullable = true)
 |-- first_time_homebuyer_flag: string (nullable = true)
 |-- maturity_date: long (nullable = true)
 |-- msa: string (nullable = true)
 |-- mi_percentage: long (nullable = true)
 |-- number_of_units: long (nullable = true)
 |-- occupancy_status: string (nullable = true)
 |-- original_cltv: long (nullable = true)
 |-- original_dti: long (nullable = true)
 |-- original_upb: long (nullable = true)
 |-- original_ltv: long (nullable = true)
 |-- original_interest_rate: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- prepayment_penalty_flag: string (nullable = true)
 |-- amortization_type: string (nullable = true)
 |-- property_state: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- loan_id: string (nullable = true)
 |-- loan_purpose: string (nullable = true)
 |-- original_loan_term: long (nullable = true)
 |-

In [13]:
origination_raw.select(
    "first_payment_date",
    "maturity_date",
).show(
    30,
    truncate=False,
)

+------------------+-------------+
|first_payment_date|maturity_date|
+------------------+-------------+
|544               |723          |
|543               |902          |
|542               |721          |
|544               |903          |
|544               |903          |
|544               |903          |
|542               |901          |
|543               |902          |
|542               |901          |
|543               |722          |
|542               |901          |
|544               |723          |
|543               |902          |
|542               |721          |
|543               |902          |
|544               |903          |
|542               |721          |
|542               |901          |
|543               |902          |
|542               |721          |
|543               |902          |
|544               |903          |
|542               |901          |
|543               |722          |
|542               |901          |
|542               |

In [14]:
print(
    "first_payment_date:",
    origination_raw.schema["first_payment_date"].dataType,
)

print(
    "maturity_date:",
    origination_raw.schema["maturity_date"].dataType,
)

first_payment_date: LongType()
maturity_date: LongType()


In [16]:
import pandas as pd
from pathlib import Path

origination_path = Path("data/02_intermediate/freddie_mac/2015/origination.parquet")

orig_pd = pd.read_parquet(origination_path)

print("Shape:", orig_pd.shape)
print("\nDtypes:")
print(
    orig_pd[
        [
            "loan_id",
            "first_payment_date",
            "maturity_date",
        ]
    ].dtypes
)

print("\nSample:")
print(
    orig_pd[
        [
            "loan_id",
            "first_payment_date",
            "maturity_date",
        ]
    ]
    .head(20)
    .to_string(index=False)
)

Shape: (1474376, 31)

Dtypes:
loan_id                  string
first_payment_date    period[M]
maturity_date         period[M]
dtype: object

Sample:
     loan_id first_payment_date maturity_date
F15Q10000001            2015-05       2030-04
F15Q10000002            2015-04       2045-03
F15Q10000003            2015-03       2030-02
F15Q10000004            2015-05       2045-04
F15Q10000005            2015-05       2045-04
F15Q10000006            2015-05       2045-04
F15Q10000007            2015-03       2045-02
F15Q10000008            2015-04       2045-03
F15Q10000009            2015-03       2045-02
F15Q10000010            2015-04       2030-03
F15Q10000011            2015-03       2045-02
F15Q10000012            2015-05       2030-04
F15Q10000013            2015-04       2045-03
F15Q10000015            2015-03       2030-02
F15Q10000016            2015-04       2045-03
F15Q10000017            2015-05       2045-04
F15Q10000018            2015-03       2030-02
F15Q10000019           

In [7]:
spark = create_spark_session(config)
model_input = spark.read.parquet("C:/Users/vorad/OneDrive/Desktop/Projects/mortgage-credit-risk/data/03_processed/behavioral/freddie_mac/2015/model-input.parquet")

c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [8]:
model_input.count()

2741047

In [1]:
import xgboost

print(xgboost.__version__)

3.4.1


In [2]:
from xgboost.spark import SparkXGBClassifier

print(SparkXGBClassifier)

<class 'xgboost.spark.estimator.SparkXGBClassifier'>


In [2]:
from pathlib import Path
import zipfile
import shutil
import re


# ============================================================
# PATHS
# ============================================================

SOURCE_ROOT = Path(
    r"C:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\data\01_raw\freddie_mac"
)

PROJECT_ROOT = Path(
    r"C:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk"
)

OUTPUT_ROOT = PROJECT_ROOT / "data" / "01_raw"


# ============================================================
# HELPERS
# ============================================================

def classify_file(filename: str) -> str | None:
    """
    Identify whether an extracted Freddie Mac file is
    origination or performance data.
    """

    name = filename.lower()

    # Freddie Mac historical files generally contain
    # "orig" and "svc" / "serv" indicators.
    if "orig" in name:
        return "orig"

    if "svcg" in name or "perf" in name or "serv" in name:
        return "perf"

    return None


def process_zip(zip_path: Path, vintage: int, quarter: str) -> None:
    """
    Extract one quarterly Freddie Mac ZIP and place the
    required files at the paths expected by the project.
    """

    output_dir = OUTPUT_ROOT / str(vintage) / quarter
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing: {zip_path.name}")
    print(f"Output:    {output_dir}")

    with zipfile.ZipFile(zip_path, "r") as z:

        members = [
            member
            for member in z.namelist()
            if not member.endswith("/")
        ]

        print("Files in ZIP:")

        for member in members:
            print(f"  {member}")

        found = {
            "orig": None,
            "perf": None,
        }

        for member in members:

            filename = Path(member).name

            file_type = classify_file(filename)

            if file_type is None:
                continue

            found[file_type] = member

        # --------------------------------------------------------
        # Validate
        # --------------------------------------------------------

        missing = [
            file_type
            for file_type, member in found.items()
            if member is None
        ]

        if missing:
            raise ValueError(
                f"Could not identify {missing} file(s) in "
                f"{zip_path.name}. Found={found}"
            )

        # --------------------------------------------------------
        # Extract + rename
        # --------------------------------------------------------

        for file_type, member in found.items():

            destination = output_dir / f"{file_type}.txt"

            if destination.exists():
                print(f"Removing existing: {destination}")
                destination.unlink()

            print(
                f"Extracting: {Path(member).name} "
                f"-> {destination.name}"
            )

            with z.open(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)

    print("Done.")


# ============================================================
# MAIN
# ============================================================

def main() -> None:

    if not SOURCE_ROOT.exists():
        raise FileNotFoundError(
            f"Source directory does not exist:\n{SOURCE_ROOT}"
        )

    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    for vintage_dir in sorted(
        SOURCE_ROOT.glob("historical_data_*")
    ):

        if not vintage_dir.is_dir():
            continue

        vintage_match = re.search(
            r"(\d{4})$",
            vintage_dir.name,
        )

        if not vintage_match:
            print(
                f"Skipping unexpected directory: "
                f"{vintage_dir.name}"
            )
            continue

        vintage = int(vintage_match.group(1))

        for zip_path in sorted(
            vintage_dir.glob("*.zip")
        ):

            quarter_match = re.search(
                r"(Q[1-4])",
                zip_path.stem,
                re.IGNORECASE,
            )

            if not quarter_match:
                print(
                    f"Skipping ZIP with unknown quarter: "
                    f"{zip_path.name}"
                )
                continue

            quarter = quarter_match.group(1).upper()

            process_zip(
                zip_path=zip_path,
                vintage=vintage,
                quarter=quarter,
            )

    print("\n========================================")
    print("Extraction completed")
    print(f"Output: {OUTPUT_ROOT}")
    print("========================================")


if __name__ == "__main__":
    main()


Processing: historical_data_2015Q1.zip
Output:    C:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\data\01_raw\2015\Q1
Files in ZIP:
  orig_2015Q1.txt
  perf_2015Q1.txt
Extracting: orig_2015Q1.txt -> orig.txt
Extracting: perf_2015Q1.txt -> perf.txt
Done.

Processing: historical_data_2015Q2.zip
Output:    C:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\data\01_raw\2015\Q2
Files in ZIP:
  orig_2015Q2.txt
  perf_2015Q2.txt
Extracting: orig_2015Q2.txt -> orig.txt
Extracting: perf_2015Q2.txt -> perf.txt
Done.

Processing: historical_data_2015Q3.zip
Output:    C:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\data\01_raw\2015\Q3
Files in ZIP:
  orig_2015Q3.txt
  perf_2015Q3.txt
Extracting: orig_2015Q3.txt -> orig.txt
Extracting: perf_2015Q3.txt -> perf.txt
Done.

Processing: historical_data_2015Q4.zip
Output:    C:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\data\01_raw\2015\Q4
Files in ZIP:
  orig_2015Q4.txt
  perf_2015Q4.txt
Extracting

In [6]:
spark = create_spark_session(config)

c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [7]:
from pyspark.sql import functions as F


df = spark.read.parquet(
    "data/03_processed/behavioral/freddie_mac/2015/model-input.parquet"
)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/c:/Users/vorad/OneDrive/Desktop/Projects/mortgage-credit-risk/notebooks/data/03_processed/behavioral/freddie_mac/2015/model-input.parquet. SQLSTATE: 42K03

In [9]:
df.columns

['loan_id',
 'observation_age',
 'period',
 'current_actual_upb',
 'current_interest_rate',
 'loan_age',
 'remaining_months_to_legal_maturity',
 'estimated_ltv',
 'current_loan_delinquency_status',
 'ddlpi',
 'modification_flag',
 'current_non_interest_bearing_upb',
 'current_interest_bearing_upb',
 'interest_rate_step_indicator',
 'payment_deferral_flag',
 'delinquency_due_to_disaster',
 'borrower_assistance_plan',
 'mi_cancellation_indicator',
 'servicer_name',
 'delinquent_accrued_interest',
 'zero_balance_code',
 'zero_balance_effective_date',
 'credit_score',
 'original_dti',
 'number_of_borrowers',
 'original_ltv',
 'original_cltv',
 'mi_percentage',
 'original_upb',
 'original_interest_rate',
 'original_loan_term',
 'first_time_homebuyer_flag',
 'property_type',
 'occupancy_status',
 'loan_purpose',
 'channel',
 'super_conforming_flag',
 'harp_indicator',
 'property_state',
 'original_dti_missing',
 'first_payment_date',
 'calculated_loan_age',
 'current_dpd_numeric',
 'current_

In [10]:
df.select("delinquent_accrued_interest").describe().show()

+-------+---------------------------+
|summary|delinquent_accrued_interest|
+-------+---------------------------+
|  count|                          0|
|   mean|                       NULL|
| stddev|                       NULL|
|    min|                       NULL|
|    max|                       NULL|
+-------+---------------------------+



In [12]:
df.select("months_since_last_paid_installment").describe().show()

+-------+----------------------------------+
|summary|months_since_last_paid_installment|
+-------+----------------------------------+
|  count|                                 0|
|   mean|                              NULL|
| stddev|                              NULL|
|    min|                              NULL|
|    max|                              NULL|
+-------+----------------------------------+



In [8]:
performance_df_test = performance_df.withColumn(
    "months_since_last_delinquency",
    F.when(F.col("ddlpi").isNull(), F.lit(None).cast("double")).otherwise(
        F.col("period").cast("double") - F.col("ddlpi").cast("double")
    ),
)

performance_df_test.select(
    "loan_id", "period", "ddlpi", "months_since_last_delinquency"
).filter(F.col("ddlpi").isNotNull()).show(30, truncate=False)

NameError: name 'performance_df' is not defined

In [19]:
performance_df.select(
    F.min("period").alias("min_period"),
    F.max("period").alias("max_period"),
    F.min("ddlpi").alias("min_ddlpi"),
    F.max("ddlpi").alias("max_ddlpi"),
    F.sum(F.when(F.col("ddlpi") > F.col("period"), 1).otherwise(0)).alias(
        "ddlpi_gt_period"
    ),
).show()

+----------+----------+---------+---------+---------------+
|min_period|max_period|min_ddlpi|max_ddlpi|ddlpi_gt_period|
+----------+----------+---------+---------+---------------+
|       540|       674|      517|      913|          36665|
+----------+----------+---------+---------+---------------+



In [20]:
performance_df.filter(F.col("ddlpi") > F.col("period")).select(
    "loan_id",
    "period",
    "ddlpi",
    "loan_age",
    "current_loan_delinquency_status",
).orderBy("period", "ddlpi",).show(50, truncate=False)

+------------+------+-----+--------+-------------------------------+
|loan_id     |period|ddlpi|loan_age|current_loan_delinquency_status|
+------------+------+-----+--------+-------------------------------+
|F15Q10080850|592   |593  |51      |00                             |
|F15Q10084893|592   |593  |50      |00                             |
|F15Q10019235|592   |593  |51      |00                             |
|F15Q10020084|592   |593  |51      |00                             |
|F15Q10120733|592   |593  |51      |00                             |
|F15Q10120954|592   |593  |50      |00                             |
|F15Q10122398|592   |593  |50      |00                             |
|F15Q10123494|592   |593  |50      |00                             |
|F15Q10037713|592   |593  |51      |00                             |
|F15Q10039856|592   |593  |51      |00                             |
|F15Q10141745|592   |593  |50      |00                             |
|F15Q10145479|592   |593  |50     

In [21]:
performance_df.filter(F.col("ddlpi") > F.col("period")).select(
    (F.col("ddlpi") - F.col("period")).alias("ddlpi_ahead")
).groupBy("ddlpi_ahead").count().orderBy(F.col("count").desc()).show(30)

+-----------+-----+
|ddlpi_ahead|count|
+-----------+-----+
|          1|31060|
|          2| 3040|
|          3|  930|
|          4|  395|
|          5|  257|
|          6|  183|
|          7|  108|
|          8|   82|
|          9|   66|
|         10|   54|
|         11|   51|
|         12|   50|
|         13|   28|
|         14|   22|
|         15|   14|
|         17|   13|
|         21|   10|
|         18|    9|
|         19|    9|
|         23|    7|
|         24|    7|
|        303|    7|
|         20|    7|
|        298|    7|
|         32|    6|
|        305|    6|
|         16|    6|
|        294|    5|
|        300|    5|
|        304|    5|
+-----------+-----+
only showing top 30 rows


In [22]:
performance_df.select(
    F.when(F.col("ddlpi").isNull(), "NULL")
    .when(F.col("ddlpi") > F.col("period"), "AHEAD")
    .otherwise("VALID")
    .alias("ddlpi_status")
).groupBy("ddlpi_status").count().show()

+------------+---------+
|ddlpi_status|    count|
+------------+---------+
|       AHEAD|    36665|
|       VALID|   681930|
|        NULL|101597986|
+------------+---------+



In [11]:
# spark = create_spark_session(config)
# origination_df = spark.read.parquet(str(os.getcwd() / origination_path))

performance_df = spark.read.parquet("C:/Users/vorad/OneDrive/Desktop/Projects/mortgage-credit-risk/data/04_model_split/behavioral/train_split.parquet")

performance_df.select("current_loan_delinquency_status").groupBy(
    "current_loan_delinquency_status"
).count().orderBy("current_loan_delinquency_status").show(100, truncate=False)

+-------------------------------+--------+
|current_loan_delinquency_status|count   |
+-------------------------------+--------+
|00                             |25571170|
|01                             |135703  |
|02                             |50225   |
+-------------------------------+--------+



In [12]:
performance_df.select("estimated_ltv").summary(
    "count",
    "min",
    "25%",
    "50%",
    "75%",
    "max",
).show()

+-------+-------------+
|summary|estimated_ltv|
+-------+-------------+
|  count|     25757098|
|    min|            1|
|    25%|           55|
|    50%|           70|
|    75%|           81|
|    max|          999|
+-------+-------------+



In [13]:
performance_df.groupBy("estimated_ltv").count().orderBy(F.desc("count")).show(
    20, truncate=False
)

+-------------+-------+
|estimated_ltv|count  |
+-------------+-------+
|999          |1480929|
|75           |643016 |
|74           |632950 |
|76           |632402 |
|73           |613318 |
|77           |604141 |
|72           |589373 |
|71           |566530 |
|78           |561630 |
|70           |547716 |
|69           |530349 |
|68           |513212 |
|79           |501708 |
|67           |497612 |
|66           |483450 |
|65           |470485 |
|64           |456872 |
|63           |443841 |
|80           |434527 |
|62           |431055 |
+-------------+-------+
only showing top 20 rows


In [14]:
performance_df.select(
    F.percentile_approx(
        "estimated_ltv",
        [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
        10000,
    ).alias("quantiles")
).show(truncate=False)

+------------------------+
|quantiles               |
+------------------------+
|[1, 51, 64, 74, 85, 999]|
+------------------------+



In [5]:
import sys
import pyspark
from pathlib import Path
from credit_risk.utils.spark import create_spark_session
from credit_risk.utils.config import read_config
config = read_config(Path.cwd().parent)
spark = create_spark_session(config)
print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("Spark:", spark.version)
print("Scala:", spark.sparkContext._jvm.scala.util.Properties.versionNumberString())

c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Python: 3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]
PySpark: 4.2.0
Spark: 4.2.0
Scala: 2.13.18


In [1]:
from pathlib import Path
from credit_risk.utils.config import read_config
from credit_risk.utils.spark import create_spark_session
from pyspark.sql import functions as F
project_path = Path.cwd().parent
config = read_config(project_path)
spark = create_spark_session(config)
training_df = spark.read.parquet(f"{project_path}/data/04_model_split/behavioral/train_split.parquet")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/04 10:31:15 WARN Utils: Your hostname, jl-vm-497045, resolves to a loopback address: 127.0.1.1; using 217.18.55.210 instead (on interface enp1s0)
26/09/04 10:31:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/ubuntu/mortgage-credit-risk/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/04 10:31:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/04 10:31:16 WARN Utils: Service

In [2]:
training_df.select(
    F.percentile_approx(
        "estimated_ltv",
        [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
        10000,
    ).alias("quantiles")
).show(truncate=False)

[Stage 1:======================================================>(455 + 2) / 457]

+------------------------+
|quantiles               |
+------------------------+
|[1, 51, 64, 73, 82, 955]|
+------------------------+

